# MemWeave 05 — Kaggle edition: recovery, feature ablation, and frozen LoCoMo validation

This Kaggle-ready follow-up rebuilds the MemWeave role-aware experiment, regenerates privacy-safe per-question results, records immutable dataset revisions and SHA-256 checksums, evaluates predeclared feature ablations, and tests frozen LongMemEval-trained models on LoCoMo.

**Before running:** in Kaggle, open the notebook's **Settings** panel and turn **Internet** on. The notebook downloads public datasets and the pinned source notebook. You do not need to upload the datasets manually unless you prefer to use a local Kaggle Dataset.

**Important:** the historical per-question CSV was not present in the repository. This notebook regenerates a new export; it does not recover the missing historical rows. The old aggregate values (0.404184 baseline and 0.709433 role-aware) are reference values only. No raw questions, answers, message text, or session text are written to the per-question export.

Run from the first cell in a fresh Kaggle session. Outputs are saved under `/kaggle/working/memweave_followup_outputs/`; use Kaggle's **Save Version** to preserve the run and download the output files.

In [4]:
# Kaggle setup. Internet must be enabled in Notebook Settings.
import os
from pathlib import Path

KAGGLE = Path('/kaggle/working')
KAGGLE.mkdir(parents=True, exist_ok=True)
os.chdir(KAGGLE)
os.environ['CONFIRM_LONGMEMEVAL_TERMS'] = 'no'   # Change to 'yes' only after reviewing LongMemEval terms.
os.environ['CONFIRM_LOCOMO_TERMS'] = 'no'        # Change to 'yes' only after reviewing LoCoMo terms.
os.environ['MEMWEAVE_OUT'] = str(KAGGLE / 'memweave_followup_outputs')
print('Working directory:', Path.cwd())
print('Output directory:', os.environ['MEMWEAVE_OUT'])


Working directory: /kaggle/working
Output directory: /kaggle/working/memweave_followup_outputs


In [1]:
import numpy
import pandas
import scipy
import sklearn
import requests
import huggingface_hub

print("All packages imported successfully!")

All packages imported successfully!


In [3]:
print("Kaggle kernel is working!")

Kaggle kernel is working!


In [2]:
%pip install -q huggingface_hub requests numpy pandas scipy scikit-learn
print('Dependencies are ready. If Kaggle displays a package-restart notice, use Session → Restart session, then run all cells from the top.')


Note: you may need to restart the kernel to use updated packages.
Dependencies are ready. If Kaggle displays a package-restart notice, use Session → Restart session, then run all cells from the top.


## 1. Resolve and hash immutable datasets

LongMemEval is resolved to an immutable Hugging Face commit before download. LoCoMo is resolved to the Git commit containing `data/locomo10.json`. If you supply a local LongMemEval file, set `LONGMEMEVAL_REVISION` and `LONGMEMEVAL_SOURCE_URL` yourself; the file SHA-256 will still be recorded, but the revision remains explicitly unknown if not supplied.

In [10]:
import os, re, json, hashlib, time, datetime, platform, sys, subprocess, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from huggingface_hub import HfApi, hf_hub_download

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def resolve_longmemeval():
    local = os.environ.get("LONGMEMEVAL_PATH")
    if local:
        return os.path.abspath(local), os.environ.get("LONGMEMEVAL_REVISION", "LOCAL_REVISION_NOT_PROVIDED"), os.environ.get("LONGMEMEVAL_SOURCE_URL", "local file; source URL not provided")
    repo_id = "xiaowu0162/longmemeval-cleaned"
    requested_revision = os.environ.get("LONGMEMEVAL_REVISION", "main")
    info = HfApi().dataset_info(repo_id, revision=requested_revision)
    commit_sha = info.sha
    path = hf_hub_download(
        repo_id=repo_id,
        filename="longmemeval_s_cleaned.json",
        repo_type="dataset",
        revision=commit_sha
    )
    url = f"https://huggingface.co/datasets/{repo_id}/resolve/{commit_sha}/longmemeval_s_cleaned.json"
    return path, commit_sha, url

LME_PATH, LME_REVISION, LME_URL = resolve_longmemeval()
LME_SHA256 = sha256_file(LME_PATH)

api_url = "https://api.github.com/repos/snap-research/locomo/commits?path=data/locomo10.json&per_page=1"
api_response = requests.get(api_url, timeout=60)
api_response.raise_for_status()
LOCOMO_COMMIT = api_response.json()[0]["sha"]
LOCOMO_URL = f"https://raw.githubusercontent.com/snap-research/locomo/{LOCOMO_COMMIT}/data/locomo10.json"
os.makedirs("data", exist_ok=True)
LOCOMO_PATH = os.path.join("data", "locomo10.json")
urllib.request.urlretrieve(LOCOMO_URL, LOCOMO_PATH)
LOCOMO_SHA256 = sha256_file(LOCOMO_PATH)

print("LongMemEval revision:", LME_REVISION)
print("LongMemEval SHA-256:", LME_SHA256)
print("LoCoMo commit:", LOCOMO_COMMIT)
print("LoCoMo SHA-256:", LOCOMO_SHA256)

if os.environ.get("CONFIRM_LONGMEMEVAL_TERMS", "").lower() != "yes":
    raise RuntimeError("Review LongMemEval terms, then set CONFIRM_LONGMEMEVAL_TERMS=yes.")
if os.environ.get("CONFIRM_LOCOMO_TERMS", "").lower() != "yes":
    raise RuntimeError("Review LoCoMo terms, then set CONFIRM_LOCOMO_TERMS=yes.")

DATA_PROVENANCE = {
    "longmemeval": {"source_url": LME_URL, "revision": LME_REVISION, "sha256": LME_SHA256},
    "locomo": {"source_url": LOCOMO_URL, "revision": LOCOMO_COMMIT, "sha256": LOCOMO_SHA256}
}


LongMemEval revision: 98d7416c24c778c2fee6e6f3006e7a073259d48f
LongMemEval SHA-256: d6f21ea9d60a0d56f34a05b609c79c88a451d2ae03597821ea3d5a9678c3a442
LoCoMo commit: cbfbc1dba6bc53d00625212a0f22d55ffee7c1fc
LoCoMo SHA-256: 79fa87e90f04081343b8c8debecb80a9a6842b76a7aa537dc9fdf651ea698ff4


## 2. Rebuild the canonical LongMemEval representation

This loads the core preprocessing and feature-building cells from the repository's final executed notebook at a pinned Git commit. It reuses the original parser, TF-IDF representation, `same_pipeline` pool, and stratified 235/235 split. The original raw JSON remains in memory only so the six role-aware features can be reconstructed.

In [11]:
# Pinned source notebook commit: the canonical executed notebook at the time this
# follow-up was authored. Update only after auditing a deliberate source change.
SOURCE_NOTEBOOK_COMMIT = "d642215226fa2a9204bb6d64c4ff0f6447920274"
SOURCE_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/tishaaaaaaaaa5-sketch/memweavee/"
    + SOURCE_NOTEBOOK_COMMIT
    + "/MemWeave/notebooks/04_MemWeave_Final_Executed.ipynb"
)
source_bytes = requests.get(SOURCE_NOTEBOOK_URL, timeout=60)
source_bytes.raise_for_status()
source_nb = source_bytes.json()

def source_cell(index):
    src = source_nb["cells"][index]["source"]
    return "".join(src) if isinstance(src, list) else str(src)

# Original setup/configuration.
exec(compile(source_cell(1), "canonical_cell_1", "exec"), globals())

# Point the original data-preparation cell at the pinned local file.
LME_PATH = LME_PATH
prep_source = source_cell(4).replace("del raw\n", "")
exec(compile(prep_source, "canonical_cell_4", "exec"), globals())

# Original base features, pools and deterministic DEV/TEST split.
exec(compile(source_cell(6), "canonical_cell_6", "exec"), globals())

POOL = "same_pipeline"
BASE_IDX = [0, 1, 7, 8, 9]
ROLE_IDX = list(range(10, 16))
FULL_IDX = BASE_IDX + ROLE_IDX
C_FROZEN = 0.3
BUDGET_FRACTION = 0.10

assert len(DEV) == 235 and len(TEST) == 235, (
    f"Expected canonical 235/235 split; got {len(DEV)}/{len(TEST)}. "
    "Do not compare this run directly with the historical aggregate."
)
assert not set(DEV).intersection(TEST)
print("Split verified:", len(DEV), "DEV /", len(TEST), "TEST")

config: {'budget': 0.1, 'budget_sweep': [0.05, 0.1, 0.2, 0.3], 'test_frac': 0.5, 'cv_folds': 5, 'C_grid': [0.03, 0.3, 3.0], 'equiv_margin': 0.03, 'alpha': 0.05, 'n_boot': 4000, 'topk': 3, 'n_random_reps': 50, 'primary_pool': 'same_pipeline', 'max_questions': None}
questions loaded: 500
unique sessions: 18464  | built in 6s
tf-idf matrix: (18464, 524288) | 22s elapsed
{'same_pipeline': 470, 'all': 470, 'filler': 470} | valid in all pools: 470
DEV: 235  TEST (locked): 235
mean pool size: {'same_pipeline': np.float64(23.4), 'all': np.float64(47.7), 'filler': np.float64(26.2)}
Split verified: 235 DEV / 235 TEST


## 3. Reconstruct the six role-aware features

Feature order is preserved from the executed experiment: maximum and top-three mean question similarity for user messages; the same two statistics for assistant messages; log user-message count; log assistant-message count. The within-question z-score matches the original feature reconstruction.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest

ROLE_FEATURE_NAMES = [
    "user_max_question_similarity",
    "user_top3_mean_question_similarity",
    "assistant_max_question_similarity",
    "assistant_top3_mean_question_similarity",
    "log_user_message_count",
    "log_assistant_message_count",
]

row_maps = {
    qi: {int(row["h"]): row for row in qrecs[qi]["rows"]}
    for qi in DEV + TEST
}

def construct_role_features_lme(qi):
    q = qrecs[qi]
    sessions = dict(zip(
        map(str, raw[qi]["haystack_session_ids"]),
        raw[qi]["haystack_sessions"]
    ))
    hashes = Q[POOL][qi]["h"]
    n = len(hashes)
    texts, roles, candidate_ids = [], [], []
    user_counts = np.zeros(n)
    assistant_counts = np.zeros(n)

    for ci, h in enumerate(hashes):
        row = row_maps[qi][int(h)]
        session = sessions[str(row["sid"])]
        for message in session:
            role = str(message.get("role", "")).lower()
            content = str(message.get("content", "")).strip()
            if role not in ("user", "assistant") or not content:
                continue
            texts.append(content)
            roles.append(role)
            candidate_ids.append(ci)
            if role == "user":
                user_counts[ci] += 1
            else:
                assistant_counts[ci] += 1

    features = np.zeros((n, 6), dtype=np.float32)
    if texts:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
        mat = vec.fit_transform([str(q["question"])] + texts)
        sims = (mat[1:] @ mat[0].T).toarray().ravel()
        grouped = {(i, role): [] for i in range(n) for role in ("user", "assistant")}
        for sim, ci, role in zip(sims, candidate_ids, roles):
            grouped[(ci, role)].append(float(sim))
        for ci in range(n):
            u = sorted(grouped[(ci, "user")], reverse=True)
            a = sorted(grouped[(ci, "assistant")], reverse=True)
            features[ci, 0] = max(u) if u else 0.0
            features[ci, 1] = np.mean(u[:3]) if u else 0.0
            features[ci, 2] = max(a) if a else 0.0
            features[ci, 3] = np.mean(a[:3]) if a else 0.0

    features[:, 4] = np.log1p(user_counts)
    features[:, 5] = np.log1p(assistant_counts)
    features = (features - features.mean(0, keepdims=True)) / (features.std(0, keepdims=True) + 1e-6)
    return features.astype(np.float32)

for qi in DEV + TEST:
    extra = construct_role_features_lme(qi)
    Q[POOL][qi]["F_aug"] = np.column_stack([Q[POOL][qi]["F"], extra]).astype(np.float32)

assert all(Q[POOL][q]["F_aug"].shape[1] == 16 for q in DEV + TEST)
print("Reconstructed", len(DEV) + len(TEST), "question feature matrices.")

Reconstructed 470 question feature matrices.


## 4. Locked-test regeneration and feature ablations

All variants use the same DEV/TEST split, `C=0.3`, content-overlap exclusion, and 10% word-count budget. Leave-one-feature-out results estimate each feature's marginal contribution conditional on the other five. Group ablations test user-side, assistant-side, similarity, and message-count groups. No test-driven selection is performed.

In [13]:
def stack_aug(qis, feature_idx):
    xs, ys, hs = [], [], []
    for qi in qis:
        d = Q[POOL][qi]
        xs.append(d["F_aug"][:, feature_idx])
        ys.append(np.asarray(d["gold"], dtype=int))
        hs.append(np.asarray(d["h"], dtype=int))
    return np.vstack(xs), np.concatenate(ys), np.concatenate(hs)

def fit_variant(feature_idx):
    held = np.fromiter(set().union(*[qrecs[q]["all_hashes"] for q in TEST]), dtype=np.int64)
    xtr, ytr, htr = stack_aug(DEV, feature_idx)
    keep = ~np.isin(htr, held)
    model = LogisticRegression(C=C_FROZEN, class_weight="balanced", max_iter=500)
    model.fit(xtr[keep], ytr[keep])
    scores = {q: model.predict_proba(Q[POOL][q]["F_aug"][:, feature_idx])[:, 1] for q in TEST}
    return model, scores, int(keep.sum()), int(len(keep))

def select_budgeted(scores, cost, budget=0.10):
    cap = budget * float(np.asarray(cost).sum())
    chosen, used = [], 0.0
    for ix in np.argsort(-np.asarray(scores), kind="stable"):
        c = float(cost[ix])
        if used + c <= cap:
            chosen.append(int(ix))
            used += c
    return np.asarray(chosen, dtype=int), used, cap

def evaluate_scores(score_map):
    out = {}
    for qi in TEST:
        d = Q[POOL][qi]
        scores = np.asarray(score_map[qi], dtype=float)
        gold = np.asarray(d["gold"], dtype=bool)
        order = np.argsort(-scores, kind="stable")
        rank_lookup = np.empty(len(order), dtype=int)
        rank_lookup[order] = np.arange(1, len(order) + 1)
        rank = int(rank_lookup[np.flatnonzero(gold)].min())
        chosen, used, cap = select_budgeted(scores, d["cost"])
        recall = float(gold[chosen].sum() / gold.sum())
        out[qi] = {"rank": rank, "recall": recall, "utilization": used / cap if cap else 0.0}
    return out

VARIANTS = {"baseline": BASE_IDX, "full_role_aware": FULL_IDX}
for j, name in enumerate(ROLE_FEATURE_NAMES):
    VARIANTS["leave_out__" + name] = [i for i in FULL_IDX if i != 10 + j]
VARIANTS["leave_out__user_message_group"] = [i for i in FULL_IDX if i not in (10, 11, 14)]
VARIANTS["leave_out__assistant_message_group"] = [i for i in FULL_IDX if i not in (12, 13, 15)]
VARIANTS["leave_out__message_similarity_group"] = [i for i in FULL_IDX if i not in (10, 11, 12, 13)]
VARIANTS["leave_out__message_count_group"] = [i for i in FULL_IDX if i not in (14, 15)]

MODELS, SCORES, METRICS, FIT_AUDIT = {}, {}, {}, {}
for name, cols in VARIANTS.items():
    model, scores, kept, total = fit_variant(cols)
    MODELS[name], SCORES[name] = model, scores
    METRICS[name] = evaluate_scores(scores)
    FIT_AUDIT[name] = {"feature_indices": cols, "training_rows_retained": kept, "training_rows_total": total}
    print(name, kept, "/", total)

baseline 4216 / 5523
full_role_aware 4216 / 5523
leave_out__user_max_question_similarity 4216 / 5523
leave_out__user_top3_mean_question_similarity 4216 / 5523
leave_out__assistant_max_question_similarity 4216 / 5523
leave_out__assistant_top3_mean_question_similarity 4216 / 5523
leave_out__log_user_message_count 4216 / 5523
leave_out__log_assistant_message_count 4216 / 5523
leave_out__user_message_group 4216 / 5523
leave_out__assistant_message_group 4216 / 5523
leave_out__message_similarity_group 4216 / 5523
leave_out__message_count_group 4216 / 5523


In [14]:
def paired_bootstrap(diff, n_boot=50000, seed=20261126):
    x = np.asarray(diff, dtype=float)
    rng = np.random.default_rng(seed)
    draws = np.empty(n_boot)
    for start in range(0, n_boot, 2000):
        end = min(start + 2000, n_boot)
        ix = rng.integers(0, len(x), size=(end-start, len(x)))
        draws[start:end] = x[ix].mean(axis=1)
    return float(x.mean()), np.quantile(draws, [0.025, 0.975]).tolist()

def exact_sign_p(diff):
    x = np.asarray(diff, dtype=float)
    pos, neg = int((x > 0).sum()), int((x < 0).sum())
    return float(binomtest(pos, pos + neg, 0.5).pvalue) if pos + neg else 1.0

def holm_adjust(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    out = np.empty(len(p))
    running = 0.0
    for rank, ix in enumerate(order):
        running = max(running, (len(p)-rank) * p[ix])
        out[ix] = min(1.0, running)
    return out

ablation_names = [n for n in VARIANTS if n.startswith("leave_out__")]
ablation_rows = []
full = np.array([METRICS["full_role_aware"][q]["recall"] for q in TEST])
for j, name in enumerate(ablation_names):
    ablated = np.array([METRICS[name][q]["recall"] for q in TEST])
    diff = full - ablated
    mean, ci = paired_bootstrap(diff, seed=20261126+j)
    ablation_rows.append({
        "ablation": name,
        "full_minus_ablated_recall": mean,
        "ci95_low": ci[0],
        "ci95_high": ci[1],
        "exact_sign_test_p": exact_sign_p(diff),
        "positive": int((diff > 0).sum()),
        "negative": int((diff < 0).sum()),
        "ties": int((diff == 0).sum())
    })
ablation_df = pd.DataFrame(ablation_rows)
ablation_df["holm_p"] = holm_adjust(ablation_df["exact_sign_test_p"].values)
display(ablation_df.round(5))

base = np.array([METRICS["baseline"][q]["recall"] for q in TEST])
gain, gain_ci = paired_bootstrap(full - base, seed=20261127)
print("Baseline recall:", base.mean())
print("Role-aware recall:", full.mean())
print("Paired gain:", gain, "95% CI:", gain_ci)
print("Historical aggregate reference only: 0.404184 -> 0.709433")

,ablation,full_minus_ablated_recall,ci95_low,ci95_high,exact_sign_test_p,positive,negative,ties,holm_p
0,leave_out__user_max_question_similarity,0.00106,-0.01418,0.01702,1.00000,3,4,228,1.00000
1,leave_out__user_top3_mean_question_similarity,-0.00638,-0.01702,0.00000,0.50000,0,2,233,1.00000
2,leave_out__assistant_max_question_similarity,-0.00142,-0.00426,0.00000,1.00000,0,1,234,1.00000
3,leave_out__assistant_top3_mean_question_simila...,-0.00638,-0.02340,0.00993,0.54883,4,7,224,1.00000
4,leave_out__log_user_message_count,0.00000,0.00000,0.00000,1.00000,0,0,235,1.00000
5,leave_out__log_assistant_message_count,0.00000,0.00000,0.00000,1.00000,0,0,235,1.00000
6,leave_out__user_message_group,0.03922,0.00929,0.07007,0.01916,20,7,208,0.17242
7,leave_out__assistant_message_group,0.02199,0.00284,0.04397,0.17957,10,4,221,1.00000
8,leave_out__message_similarity_group,0.42071,0.36248,0.47872,0.00000,126,6,103,0.00000
9,leave_out__message_count_group,0.00000,0.00000,0.00000,1.00000,0,0,235,1.00000


Baseline recall: 0.40418439716312055
Role-aware recall: 0.8206382978723404
Paired gain: 0.4164539007092199 95% CI: [0.3587234042553191, 0.4741843971631206]
Historical aggregate reference only: 0.404184 -> 0.709433


## 5. Regenerate privacy-safe per-question export and manifest

The export contains a truncated SHA-256 of each question identifier, question type, candidate/gold counts, ranks, budgeted recall, and budget utilization. It contains no cleartext question IDs, question text, answers, raw messages, or session content.

In [15]:
OUT_DIR = os.environ.get("MEMWEAVE_OUT", "memweave_followup_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

def qid_hash(qid):
    return hashlib.sha256(str(qid).encode("utf-8")).hexdigest()[:20]

export_rows = []
for qi in TEST:
    row = {
        "question_id_sha256_20": qid_hash(qrecs[qi]["qid"]),
        "question_type": qrecs[qi]["qtype"],
        "candidate_count": int(len(Q[POOL][qi]["gold"])),
        "gold_session_count": int(np.asarray(Q[POOL][qi]["gold"]).sum())
    }
    for variant in VARIANTS:
        m = METRICS[variant][qi]
        row[variant + "__best_gold_rank"] = m["rank"]
        row[variant + "__budgeted_evidence_recall"] = m["recall"]
        row[variant + "__budget_utilization"] = m["utilization"]
    row["full_minus_baseline_budget_recall"] = (
        METRICS["full_role_aware"][qi]["recall"] - METRICS["baseline"][qi]["recall"]
    )
    export_rows.append(row)

per_question = pd.DataFrame(export_rows)
per_question_path = os.path.join(OUT_DIR, "memweave_role_aware_per_question_privacy_safe.csv")
per_question.to_csv(per_question_path, index=False)
ablation_path = os.path.join(OUT_DIR, "role_feature_ablation.csv")
ablation_df.to_csv(ablation_path, index=False)

manifest = {
    "experiment": "MemWeave role-aware recovery and ablation follow-up",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source_notebook_commit": SOURCE_NOTEBOOK_COMMIT,
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or "not_available",
    "python": sys.version,
    "platform": platform.platform(),
    "datasets": DATA_PROVENANCE,
    "split": {"seed": SEED, "dev_questions": len(DEV), "test_questions": len(TEST), "pool": POOL},
    "model": {"class": "LogisticRegression", "C": C_FROZEN, "class_weight": "balanced"},
    "budget": {"fraction": BUDGET_FRACTION, "unit": "pool word count, not tokenizer tokens"},
    "baseline_feature_indices": BASE_IDX,
    "role_feature_names": ROLE_FEATURE_NAMES,
    "role_feature_indices": ROLE_IDX,
    "fit_audit": FIT_AUDIT,
    "primary_metrics": {
        "baseline_recall": float(base.mean()),
        "role_aware_recall": float(full.mean()),
        "paired_gain": float(gain),
        "paired_bootstrap_95_ci": gain_ci
    },
    "per_question_export": os.path.basename(per_question_path),
    "per_question_export_sha256": sha256_file(per_question_path),
    "ablation_export_sha256": sha256_file(ablation_path),
    "privacy_note": "No raw question text, answers, message text, session text, or cleartext question IDs exported.",
    "limitations": [
        "This regenerates results; it does not recover the missing historical CSV.",
        "If dataset revision/hash differs from the historical source, this is not a byte-identical reproduction.",
        "Question-level inference is not conversation-clustered.",
        "Ablations estimate conditional marginal contribution, not causal effects.",
        "Evidence recall is not end-to-end answer accuracy."
    ]
}
manifest_path = os.path.join(OUT_DIR, "memweave_followup_run_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved:", per_question_path)
print("Per-question SHA-256:", manifest["per_question_export_sha256"])
print("Saved:", ablation_path)
print("Saved:", manifest_path)

Saved: /kaggle/working/memweave_followup_outputs/memweave_role_aware_per_question_privacy_safe.csv
Per-question SHA-256: fbe3c234c6038d0918dc1b5f8648fdb39fecd790df5fb52bdb6f7743b9b6d82a
Saved: /kaggle/working/memweave_followup_outputs/role_feature_ablation.csv
Saved: /kaggle/working/memweave_followup_outputs/memweave_followup_run_manifest.json


## 6. Freeze LongMemEval models, then evaluate LoCoMo

Both transfer models are fit on LongMemEval DEV only. LoCoMo is never used for fitting, tuning, calibration, or feature selection. The two speaker-role orientations are both reported because the `speaker_a`/`speaker_b` mapping is arbitrary. Conversation-level means are bootstrapped by conversation cluster.

In [17]:

# ============================================================
# FREEZE MODELS + INDEPENDENT LOCOMO EVALUATION
# ============================================================

import json
import re
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer


# ------------------------------------------------------------
# 1. FREEZE MODELS USING LONGMEMEVAL DEV DATA ONLY
# ------------------------------------------------------------

FROZEN_BASELINE = LogisticRegression(
    C=C_FROZEN,
    class_weight="balanced",
    max_iter=500
)

Xb, yb, _ = stack_aug(DEV, BASE_IDX)
FROZEN_BASELINE.fit(Xb, yb)


FROZEN_ROLE_AWARE = LogisticRegression(
    C=C_FROZEN,
    class_weight="balanced",
    max_iter=500
)

Xa, ya, _ = stack_aug(DEV, FULL_IDX)
FROZEN_ROLE_AWARE.fit(Xa, ya)

print("Frozen LongMemEval models fitted successfully.")


# ------------------------------------------------------------
# 2. LOAD LOCOMO
# ------------------------------------------------------------

with open(LOCOMO_PATH, "r", encoding="utf-8") as f:
    locomo = json.load(f)

if isinstance(locomo, dict):
    if "data" in locomo:
        locomo = locomo["data"]
    else:
        locomo = [locomo]

if not isinstance(locomo, list):
    raise ValueError("Unexpected LoCoMo dataset structure.")

print("LoCoMo conversations loaded:", len(locomo))


# ------------------------------------------------------------
# 3. ROLE-AWARE SESSION FEATURES
# ------------------------------------------------------------

def role_features_for_sessions(question, sessions):

    n = len(sessions)

    texts = []
    roles = []
    cand = []

    uc = np.zeros(n)
    ac = np.zeros(n)

    for ci, turns in enumerate(sessions):

        for role, text in turns:

            text = str(text).strip()

            if not text or role not in ("user", "assistant"):
                continue

            texts.append(text)
            roles.append(role)
            cand.append(ci)

            if role == "user":
                uc[ci] += 1
            else:
                ac[ci] += 1

    f = np.zeros((n, 6), dtype=np.float32)

    if texts:

        vec = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            sublinear_tf=True
        )

        mat = vec.fit_transform(
            [str(question)] + texts
        )

        sims = (
            mat[1:] @ mat[0].T
        ).toarray().ravel()

        grouped = {
            (i, r): []
            for i in range(n)
            for r in ("user", "assistant")
        }

        for sim, ci, role in zip(sims, cand, roles):
            grouped[(ci, role)].append(float(sim))

        for ci in range(n):

            u = sorted(
                grouped[(ci, "user")],
                reverse=True
            )

            a = sorted(
                grouped[(ci, "assistant")],
                reverse=True
            )

            f[ci, 0] = max(u) if u else 0.0
            f[ci, 1] = np.mean(u[:3]) if u else 0.0

            f[ci, 2] = max(a) if a else 0.0
            f[ci, 3] = np.mean(a[:3]) if a else 0.0

    f[:, 4] = np.log1p(uc)
    f[:, 5] = np.log1p(ac)

    return (
        (f - f.mean(0, keepdims=True)) /
        (f.std(0, keepdims=True) + 1e-6)
    ).astype(np.float32)


# ------------------------------------------------------------
# 4. EXTRACT GOLD EVIDENCE SESSION INDICES
# ------------------------------------------------------------

def locomo_gold_indices(qa, session_numbers):

    gold = set()

    for evidence in qa.get("evidence", []) or []:

        for match in re.findall(
            r"D(\d+):",
            str(evidence)
        ):

            number = int(match)

            if number in session_numbers:
                gold.add(session_numbers.index(number))

    return gold


# ------------------------------------------------------------
# 5. SAFE SESSION EXTRACTION
# ------------------------------------------------------------

def extract_session_messages(conv, number):

    key = f"session_{number}"
    messages = conv.get(key, [])

    if isinstance(messages, dict):
        messages = [messages]

    if not isinstance(messages, list):
        return []

    turns = []

    speaker_a = conv.get("speaker_a")

    for message in messages:

        if not isinstance(message, dict):
            continue

        text = str(message.get("text", "")).strip()

        if not text:
            continue

        speaker = message.get("speaker")

        # Map the two original speakers to the two roles.
        if speaker == speaker_a:
            role = "user"
        else:
            role = "assistant"

        turns.append((role, text))

    return turns


# ------------------------------------------------------------
# 6. EVALUATE ONE SPEAKER ORIENTATION
# ------------------------------------------------------------

def evaluate_locomo_orientation(speaker_a_as_user=True):

    records = []

    skipped_empty = 0
    skipped_invalid = 0
    processed = 0

    for ci, item in enumerate(locomo):

        if not isinstance(item, dict):
            skipped_invalid += 1
            continue

        conv = item.get("conversation", {})

        if not isinstance(conv, dict):
            skipped_invalid += 1
            continue

        session_numbers = sorted(
            int(k.split("_")[1])
            for k in conv
            if re.fullmatch(r"session_\d+", k)
        )

        if not session_numbers:
            skipped_invalid += 1
            continue

        sessions = []
        full_texts = []
        words = []
        nuser = []
        nturns = []
        fp = []
        digit = []

        for number in session_numbers:

            turns = extract_session_messages(
                conv,
                number
            )

            # Respect the requested speaker orientation.
            if not speaker_a_as_user:
                turns = [
                    (
                        "assistant" if role == "user" else "user",
                        text
                    )
                    for role, text in turns
                ]

            sessions.append(turns)

            full, w, uw, nt, fpv, dgv = session_stats(
                turns,
                "user"
            )

            full = str(full).strip()

            full_texts.append(full)
            words.append(w)
            nuser.append(uw)
            nturns.append(nt)
            fp.append(fpv)
            digit.append(dgv)

        # ----------------------------------------------------
        # IMPORTANT FIX:
        # Do not send an empty document collection to
        # HashingVectorizer / TfidfTransformer.
        # ----------------------------------------------------

        if not full_texts or not any(full_texts):

            skipped_empty += 1

            print(
                f"Skipping conversation {ci}: "
                "no usable session text."
            )

            continue

        valid_indices = [
            i for i, text in enumerate(full_texts)
            if text.strip()
        ]

        if not valid_indices:

            skipped_empty += 1
            continue

        # Keep only sessions with usable text.
        sessions = [sessions[i] for i in valid_indices]
        full_texts = [full_texts[i] for i in valid_indices]
        words = [words[i] for i in valid_indices]
        nuser = [nuser[i] for i in valid_indices]
        nturns = [nturns[i] for i in valid_indices]
        fp = [fp[i] for i in valid_indices]
        digit = [digit[i] for i in valid_indices]

        session_numbers = [
            session_numbers[i] for i in valid_indices
        ]

        # ----------------------------------------------------
        # TRANSFORM SESSION TEXT
        # ----------------------------------------------------

        V = tfidf.transform(
            hv.transform(full_texts)
        ).astype(np.float32).tocsr()

        conv_q = []

        for qa in item.get("qa", []) or []:

            if not isinstance(qa, dict):
                continue

            if qa.get("category") == 5:
                continue

            gold = locomo_gold_indices(
                qa,
                session_numbers
            )

            if gold:
                conv_q.append((qa, gold))

        if not conv_q:
            continue

        method_scores = {
            k: []
            for k in [
                "frozen_baseline",
                "frozen_role_aware",
                "recency",
                "low_novelty",
                "random"
            ]
        }

        # ----------------------------------------------------
        # QUESTION EVALUATION
        # ----------------------------------------------------

        for qi, (qa, gold) in enumerate(conv_q):

            days = np.arange(
                len(session_numbers),
                dtype=float
            )

            Fbase, S = build_features(
                V,
                np.asarray(words, float),
                np.asarray(nuser, float),
                np.asarray(nturns, float),
                np.asarray(fp, float),
                np.asarray(digit, float),
                days,
                None
            )

            Frole = role_features_for_sessions(
                qa.get("question", ""),
                sessions
            )

            Ffull = np.column_stack(
                [Fbase, Frole]
            ).astype(np.float32)

            # ------------------------------------------------
            # MODEL AND BASELINE SCORES
            # ------------------------------------------------

            scores = {

                "frozen_baseline":
                    FROZEN_BASELINE.predict_proba(
                        Fbase[:, BASE_IDX]
                    )[:, 1],

                "frozen_role_aware":
                    FROZEN_ROLE_AWARE.predict_proba(
                        Ffull[:, FULL_IDX]
                    )[:, 1],

                "recency":
                    Fbase[:, 2],

                "low_novelty":
                    -Fbase[:, 7],

                "random":
                    np.random.default_rng(
                        9000 + ci * 100 + qi
                    ).random(len(words))
            }

            # ------------------------------------------------
            # BUDGETED RETENTION AND EVIDENCE RECALL
            # ------------------------------------------------

            for name, values in scores.items():

                selected, _, _ = select_budgeted(
                    values,
                    words
                )

                selected = set(
                    np.asarray(selected).tolist()
                )

                recall = (
                    len(selected & gold) / len(gold)
                )

                method_scores[name].append(recall)

        records.append({

            "conversation_index": ci,

            "n_questions": len(conv_q),

            **{
                name: float(np.mean(values))
                if values else np.nan

                for name, values in method_scores.items()
            }

        })

        processed += 1

    print("\nEvaluation diagnostics")
    print("----------------------")
    print("Processed conversations:", processed)
    print("Skipped empty conversations:", skipped_empty)
    print("Skipped invalid conversations:", skipped_invalid)

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 7. RUN BOTH SPEAKER ORIENTATIONS
# ------------------------------------------------------------

print("\nRunning orientation A...")
LOCOMO_A_USER = evaluate_locomo_orientation(True)

print("\nRunning orientation B...")
LOCOMO_B_USER = evaluate_locomo_orientation(False)


# ------------------------------------------------------------
# 8. RESULTS SUMMARY
# ------------------------------------------------------------

print("\nLoCoMo evaluation completed.")

print(
    "Orientation A conversation clusters:",
    len(LOCOMO_A_USER)
)

print(
    "Orientation B conversation clusters:",
    len(LOCOMO_B_USER)
)

print("\nOrientation A results:")
display(LOCOMO_A_USER)

print("\nOrientation B results:")
display(LOCOMO_B_USER)


Frozen LongMemEval models fitted successfully.
LoCoMo conversations loaded: 10

Running orientation A...

Evaluation diagnostics
----------------------
Processed conversations: 10
Skipped empty conversations: 0
Skipped invalid conversations: 0

Running orientation B...

Evaluation diagnostics
----------------------
Processed conversations: 10
Skipped empty conversations: 0
Skipped invalid conversations: 0

LoCoMo evaluation completed.
Orientation A conversation clusters: 10
Orientation B conversation clusters: 10

Orientation A results:


,conversation_index,n_questions,frozen_baseline,frozen_role_aware,recency,low_novelty,random
0,0,150,0.061667,0.468000,0.110000,0.073889,0.115222
1,1,81,0.024691,0.570988,0.061728,0.024691,0.009259
2,2,152,0.134868,0.576206,0.100329,0.098136,0.093202
3,3,199,0.086404,0.596151,0.129564,0.114824,0.065466
4,4,178,0.089513,0.736102,0.120787,0.082638,0.081661
5,5,123,0.096883,0.620267,0.136334,0.073171,0.107995
6,6,150,0.141556,0.750000,0.111333,0.076667,0.107111
7,7,191,0.061394,0.678322,0.064011,0.051795,0.126839
8,8,156,0.097505,0.564911,0.068429,0.089919,0.111806
9,9,156,0.140491,0.608440,0.093483,0.140491,0.072650



Orientation B results:


,conversation_index,n_questions,frozen_baseline,frozen_role_aware,recency,low_novelty,random
0,0,150,0.061667,0.488000,0.110000,0.073889,0.115222
1,1,81,0.024691,0.549383,0.061728,0.024691,0.009259
2,2,152,0.134868,0.623904,0.100329,0.098136,0.093202
3,3,199,0.086404,0.596151,0.129564,0.114824,0.065466
4,4,178,0.089513,0.723930,0.120787,0.082638,0.081661
5,5,123,0.096883,0.591812,0.136334,0.073171,0.107995
6,6,150,0.141556,0.746667,0.111333,0.076667,0.107111
7,7,191,0.061394,0.678322,0.064011,0.051795,0.126839
8,8,156,0.097505,0.563736,0.068429,0.089919,0.111806
9,9,156,0.140491,0.603098,0.093483,0.140491,0.072650


In [18]:
def cluster_bootstrap_ci(diff, n_boot=20000, seed=20261128):
    x = np.asarray(diff, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    means = x[idx].mean(axis=1)
    return float(x.mean()), np.quantile(means, [0.025, 0.975]).tolist()

locomo_rows = []
for orientation, frame in [
    ("speaker_a_as_user", LOCOMO_A_USER),
    ("speaker_b_as_user", LOCOMO_B_USER)
]:
    for comparator in ["frozen_baseline", "recency", "low_novelty", "random"]:
        delta = frame["frozen_role_aware"].to_numpy() - frame[comparator].to_numpy()
        mean, ci = cluster_bootstrap_ci(delta, seed=20261128 + len(locomo_rows))
        locomo_rows.append({
            "role_orientation": orientation,
            "comparison": "frozen_role_aware_minus_" + comparator,
            "n_conversations": len(frame),
            "role_aware_mean_recall": float(frame["frozen_role_aware"].mean()),
            "comparator_mean_recall": float(frame[comparator].mean()),
            "mean_difference": mean,
            "cluster_bootstrap_ci95_low": ci[0],
            "cluster_bootstrap_ci95_high": ci[1]
        })

locomo_summary = pd.DataFrame(locomo_rows)
display(locomo_summary.round(5))

LOCOMO_A_USER.to_csv(os.path.join(OUT_DIR, "locomo_per_conversation_speaker_a_user.csv"), index=False)
LOCOMO_B_USER.to_csv(os.path.join(OUT_DIR, "locomo_per_conversation_speaker_b_user.csv"), index=False)
locomo_summary.to_csv(os.path.join(OUT_DIR, "locomo_frozen_validation_summary.csv"), index=False)

manifest["locomo_validation"] = {
    "dataset": DATA_PROVENANCE["locomo"],
    "models_frozen_before_scoring": True,
    "training_data": "LongMemEval DEV only",
    "locomo_fitting_or_tuning": False,
    "role_orientations": ["speaker_a_as_user", "speaker_b_as_user"],
    "inference_unit": "conversation cluster",
    "n_conversations": int(len(LOCOMO_A_USER)),
    "summary_file": "locomo_frozen_validation_summary.csv"
}
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print("LoCoMo artifacts and updated manifest saved in:", OUT_DIR)

,role_orientation,comparison,n_conversations,role_aware_mean_recall,comparator_mean_recall,mean_difference,cluster_bootstrap_ci95_low,cluster_bootstrap_ci95_high
0,speaker_a_as_user,frozen_role_aware_minus_frozen_baseline,10,0.61694,0.09350,0.52344,0.47717,0.57130
1,speaker_a_as_user,frozen_role_aware_minus_recency,10,0.61694,0.09960,0.51734,0.46782,0.56782
2,speaker_a_as_user,frozen_role_aware_minus_low_novelty,10,0.61694,0.08262,0.53432,0.48141,0.58890
3,speaker_a_as_user,frozen_role_aware_minus_random,10,0.61694,0.08912,0.52782,0.47410,0.57801
4,speaker_b_as_user,frozen_role_aware_minus_frozen_baseline,10,0.61650,0.09350,0.52300,0.48287,0.56638
5,speaker_b_as_user,frozen_role_aware_minus_recency,10,0.61650,0.09960,0.51690,0.46977,0.56425
6,speaker_b_as_user,frozen_role_aware_minus_low_novelty,10,0.61650,0.08262,0.53388,0.48632,0.58402
7,speaker_b_as_user,frozen_role_aware_minus_random,10,0.61650,0.08912,0.52738,0.47844,0.57290


LoCoMo artifacts and updated manifest saved in: /kaggle/working/memweave_followup_outputs


## 7. Reporting checklist

- [ ] Confirm the LongMemEval and LoCoMo licenses/terms.
- [ ] Confirm recorded immutable revisions and SHA-256 values.
- [ ] Compare the new LongMemEval aggregate with the historical reference, but do not force a match if the dataset revision differs.
- [ ] Inspect all leave-one-out and group ablation confidence intervals and Holm-adjusted p-values.
- [ ] Treat nonsignificant or wide-interval ablations as inconclusive.
- [ ] Report both LoCoMo speaker orientations; do not choose the better one post hoc.
- [ ] State clearly that evidence recall is not answer accuracy and that the original missing per-question export was regenerated, not recovered.